# ProtVS Quickstart

**ProtVS** predicts protein localization images from reference microscopy channels
(nucleus, ER, microtubules) using a conditional diffusion model.

**Workflow:**
1. Download pre-trained checkpoints (first time only)
2. Load a reference cell image
3. Explore available proteins and cell lines
4. Predict protein localization
5. Visualize and save results

## Setup

In [ ]:
import numpy as np
from tifffile import imread

from protvs import ProtVS

## Step 1: Download Checkpoints

Downloads model weights into the `protvs/` package directory.
Skip this cell if you have already downloaded them.

In [ ]:
ProtVS.download_checkpoints()

## Step 2: Initialize Model

In [ ]:
model = ProtVS()
print(model.summary())

## Step 3: Explore Available Proteins and Cell Lines

Check what proteins and cell lines the model supports before predicting.

In [ ]:
print(f"Proteins available: {len(model.available_proteins)}")
print("First 10:", model.available_proteins[:10])

print(f"\nCell lines available: {len(model.available_cell_lines)}")
print(model.available_cell_lines)

## Step 4: Load a Reference Image

The model expects a reference cell image with shape `[H, W, 3]` or `[H, W, 4]`,
pixel values in `[0, 1]` or `[0, 255]`.
For 4-channel TIFFs, channels 0, 2, and 3 are used as conditioning
(nucleus, ER, microtubules); channel 1 is ignored.

In [ ]:
# Replace with the path to your reference image
REF_PATH = "data/reference_cell.tiff"

ref_image = imread(REF_PATH)
print(f"Image shape: {ref_image.shape}, dtype: {ref_image.dtype}")

## Step 5: Predict Protein Localization

Pass one or more `(image, protein, cell_line)` combinations in a single call.
Each entry in the lists corresponds to one prediction.

In [ ]:
results = model.predict(
    images=[ref_image, ref_image],
    protein_names=["ACTB", "TUBB"],
    cell_line_names=["U-2 OS", "U-2 OS"],
    num_inference_steps=50,   # lower (e.g. 20) for faster, less accurate results
    seed=42,                  # set for reproducibility
)

print(f"Predicted {len(results)} image(s)")
print(f"Output shape per image: {results[0].shape}")

## Step 6: Visualize Results

In [ ]:
results.show_prediction()

## Step 7: Save Results

Predictions are saved as uint8 TIFFs named
`{prefix}_{index}_{cell_line}_cell_{protein}.tif`.

In [ ]:
results.save_prediction(
    prefix="quickstart",
    directory="predictions/",
)
print("Saved to predictions/")

## Next Steps

**Predict many proteins at once** — pass lists of any length to `model.predict()`;
use `batch_size` to control GPU memory usage:
```python
results = model.predict(
    images=[ref] * len(proteins),
    protein_names=proteins,
    cell_line_names=["U-2 OS"] * len(proteins),
    batch_size=8,
)
```

**Fine-tune on your own data** — provide a directory of 4-channel TIFFs:
```python
model.fit(
    image_dir="data/train",
    image_files=["cell_0.tiff", "cell_1.tiff"],
    protein_names=["CDT1", "TUBB"],
    cell_line_names=["U-2 OS", "U-2 OS"],
    output_dir="finetuned/",
    num_epochs=50,
)
```

**Save and reload a fine-tuned model:**
```python
model.save("finetuned/")
model = ProtVS(checkpoint_dir="finetuned/")
```